In [0]:

import hashlib
import re
import csv

from pathlib import Path
from pyspark.sql import functions as F


TABLE_NAME = "workspace.aerosignal_bronze.flights_raw"

REQUIRED_COLUMNS = [
    "YEAR",
    "MONTH",
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "DEP_DELAY",
    "ARR_DELAY",
    "CANCELLED",
    "DIVERTED",
]


def ingest_bts_month(
    source_path: str,
    year: int,
    month: int,
    source_sha256: str,
) -> int:

    # 1. Validate parameters
    if not isinstance(year, int) or not 1987 <= year <= 2100:
        raise ValueError("Invalid year")

    if not isinstance(month, int) or not 1 <= month <= 12:
        raise ValueError("Month must be between 1 and 12")

    if not re.fullmatch(r"[a-fA-F0-9]{64}", source_sha256):
        raise ValueError("Invalid SHA256")

    # 2. Verify source file checksum
    digest = hashlib.sha256()

    with open(source_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    if digest.hexdigest() != source_sha256.lower():
        raise ValueError("Source SHA256 mismatch")

    # 3. Read raw CSV
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .csv(source_path)
    )

    # 4. Validate required columns
    missing = set(REQUIRED_COLUMNS) - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    # 5. Validate source rows
    row_count = df.count()

    if row_count == 0:
        raise ValueError("Source file is empty")

    flight_date = F.to_date(
    F.col("FL_DATE"),
    "M/d/yyyy h:mm:ss a"
    )

    invalid_rows = df.filter(
        F.col("YEAR").cast("int").isNull()
        | (F.col("YEAR").cast("int") != year)
        | F.col("MONTH").cast("int").isNull()
        | (F.col("MONTH").cast("int") != month)
        | flight_date.isNull()
        | (F.year(flight_date) != year)
        | (F.month(flight_date) != month)
    )

    if invalid_rows.limit(1).count() > 0:
        raise ValueError("CSV contains invalid dates or wrong month")

    # 6. Add ingestion metadata
    bronze_df = (
        df
        .withColumn("_source_file", F.lit(Path(source_path).name))
        .withColumn("_source_sha256", F.lit(source_sha256.lower()))
        .withColumn("_ingested_at_utc", F.current_timestamp())
        .withColumn("_source_year", F.lit(year))
        .withColumn("_source_month", F.lit(month))
    )

    # 7. Save to Delta
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .partitionBy("_source_year", "_source_month")
            .saveAsTable(TABLE_NAME)
        )
    else:
        predicate = (
            f"_source_year = {year} AND _source_month = {month}"
        )

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", predicate)
            .saveAsTable(TABLE_NAME)
        )

    return row_count


In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

source_path = (
    "/Volumes/workspace/aerosignal_bronze/"
    "source_files/T_ONTIME_REPORTING.csv"
)

#Calculate the actual SHA256 of the uploaded file
with open(source_path, "rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

rows = ingest_bts_month(
    source_path=source_path,
    year=2025,
    month=1,
    source_sha256=source_sha256,
)

print(f"Successfully ingested {rows} rows")

In [0]:
display(
    spark.sql("""
        SELECT
            _source_year,
            _source_month,
            COUNT(*) AS total_rows
        FROM workspace.aerosignal_bronze.flights_raw
        GROUP BY _source_year, _source_month
        ORDER BY _source_year, _source_month
    """)
)

In [0]:
%sql
SELECT
    _source_year,
    _source_month,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT _source_sha256) AS source_versions
FROM workspace.aerosignal_bronze.flights_raw
WHERE _source_year = 2025
    AND _source_month = 1
GROUP BY _source_year, _source_month;

In [0]:
rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f" DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]


try:
    # Call ingestion with wrong month
    ingest_bts_month(
        source_path="/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv",
        year=2025,
        month=2,
        source_sha256=source_sha256
    )
except ValueError as e:
    print("Expected error:", e)
else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f" DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Version before:", version_before)
print("Version after:", version_after)

assert rows_before == rows_after
assert version_before == version_after

In [0]:
rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

try:
    ingest_bts_month(
        source_path="/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv",
        year=2025,
        month=1,
        source_sha256="0" * 64
    )
except ValueError as e:
    print("Expected error:", e)
else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

assert rows_before == rows_after
assert version_before == version_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Version before", version_before)
print("Version after", version_after)

print("TEST PASSED")


In [0]:
source_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/T_ONTIME_REPORTING.csv"
)

test_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/test_missing_column.csv"
)

with open(source_path, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    columns = [
        col for col in reader.fieldnames
        if col != "FL_DATE"
    ]

    first_record = next(reader)


with open(test_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=columns,
        extrasaction="ignore"
    )

    writer.writeheader()
    writer.writerow(first_record)

print("Test CSV created successfully")

print("FL_DATE exists:", "FL_DATE" in columns)

with open(test_path, "rb") as file:
    test_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

print("SHA256:", test_sha256)

rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

try:
    ingest_bts_month(
        source_path=test_path,
        year=2025,
        month=1,
        source_sha256=test_sha256
    )

except ValueError as e:
    print("Expected error:", e)

    assert "Missing columns" in str(e)
    assert "FL_DATE" in str(e)

else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

print("Rows before:", rows_before)
print("Version before:", version_before)
print("Rows after:", rows_after)
print("Version after:", version_after)

assert rows_before == rows_after
assert version_before == version_after

print("TEST PASSED")

In [0]:
test_path = (
    "/Volumes/workspace/aerosignal_bronze/source_files/test_empty.csv"
)

with open(test_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(REQUIRED_COLUMNS)

with open(test_path, "rb") as file:
    test_sha256 = hashlib.file_digest(
        file, "sha256"
    ).hexdigest()

print("SHA256:", test_sha256)

rows_before = spark.table(TABLE_NAME).count()

version_before = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

try:
    ingest_bts_month(
        source_path=test_path,
        year=2025,
        month=1,
        source_sha256=test_sha256
    )

except ValueError as e:
    assert str(e) == "Source file is empty"
    print("Expected error", e)

else:
    raise AssertionError("Invalid input was accepted")

rows_after = spark.table(TABLE_NAME).count()

version_after = spark.sql(
    f"DESCRIBE HISTORY {TABLE_NAME}"
).first()["version"]

print("Rows before:", rows_before)
print("Version before:", version_before)
print("Rows after:", rows_after)
print("Version after:", version_after)

assert rows_before == rows_after
assert version_before == version_after

print("TEST PASSED")